### NOTEBOOK 3: CROSS-DATASET EVALUATION & GENERALIZATION ANALYSIS

This notebook answers the critical question: **"Will my models work on data they've never seen before?"** 

It tests trained models on completely different datasets to measure real-world generalization capability.

#### Pipeline Overview

1. **Load Trained Models** (`isolation_forest.pkl`, `xgboost_model.pkl`, `lstm_autoencoder.pt`)

2. **Define LSTM-Autoencoder Architecture**
   - Recreate the neural network structure to load saved weights
   - Ensures model compatibility for inference

3. **Load Cross-Dataset Samples**
   - Load preprocessed data from NEW datasets NOT used in training
   - These come from different hardware, protocols, and network environments

4. **Cross-Dataset Evaluation Engine**
   - Run ALL THREE models on each new dataset
   - Measure: Accuracy, Precision, Recall, F1-Score
   - Compare performance across all datasets

5. **Generalization Gap Analysis**
   - Calculate performance drop when moving to new data
   - Formula: Gap = Max F1 - Min F1 across datasets
   - Interpretation:
     - Gap < 0.10 → Excellent generalization
     - Gap 0.10-0.25 → Moderate generalization  
     - Gap > 0.25 → Poor generalization (model memorized training data)

6. **Visualization & Reporting**
   - Generate performance comparison tables
   - Create F1-score heatmaps
   - Visualize generalization gaps

#### Outputs

| File | Description |
|------|-------------|
| `cross_dataset_evaluation.csv` | Complete performance metrics table |
| `cross_dataset_f1_comparison.png` | Bar chart comparing models across datasets |
| `f1_score_heatmap.png` | Heatmap of F1-scores (Models vs Datasets) |
| `generalization_gap.png` | Gap analysis visualization |

Supervised Models (e.g., Random Forest, XGBoost): Yield class outputs (e.g., 0 for Normal Traffic, 1 for Attack/Anomaly).

Unsupervised Models (e.g., Isolation Forest, One-Class SVM): Yield outlier scores (1 for Inliers/Normal, -1 for Outliers/Anomalies).

**Output :**

    Evaluation Data: cross_dataset_evaluation.csv  
    
    Visualizations: cross_dataset_f1_comparison.png  ; f1_score_heatmap.png  ; generalization_gap.png

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
import os

from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, accuracy_score
)

import torch
import torch.nn as nn

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Cross-Dataset Evaluation Environment Loaded")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

Cross-Dataset Evaluation Environment Loaded
NumPy version: 2.4.2
PyTorch version: 2.11.0+cpu


#### SECTION 1: LOAD TRAINED MODELS

In [4]:
print("SECTION 1: LOADING TRAINED MODELS")
print("="*70)

try:
    iso_forest = joblib.load("../outputs/isolation_forest.pkl")
    xgb_model = joblib.load("../outputs/xgboost_model.pkl")
    print("\nSuccessfully loaded trained models:")
    print("  - isolation_forest.pkl")
    print("  - xgboost_model.pkl")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("Please run Notebook 2 (2-model_trained.ipynb) first.")
    raise

# Load model configuration
try:
    with open("../outputs/model_config.json", 'r') as f:
        model_config = json.load(f)
    print("  - model_config.json")
except FileNotFoundError:
    print("WARNING: model_config.json not found. Using defaults.")
    model_config = {
        'seq_len': 10,
        'num_features': 7,
        'threshold': 0.05,
        'threshold_percentile': 95,
        'hidden_dim': 32,
        'latent_dim': 16
    }

print(f"\nModel Configuration:")
for key, value in model_config.items():
    print(f"  {key}: {value}")

SECTION 1: LOADING TRAINED MODELS

Successfully loaded trained models:
  - isolation_forest.pkl
  - xgboost_model.pkl
  - model_config.json

Model Configuration:
  seq_len: 10
  num_features: 73
  threshold: 1.131716251373291
  threshold_percentile: 90
  hidden_dim: 64
  latent_dim: 16
  num_classes: 5
  class_names: ['Normal / Benign', 'MITM / PLC Attack', 'Port Scan / Reconnaissance', 'Telnet PLC Attack', 'Web Access Attack']
  attack_types: {'0': 'Normal / Benign', '1': 'MITM / PLC Attack', '2': 'Port Scan / Reconnaissance', '3': 'Telnet PLC Attack', '4': 'Web Access Attack', '5': 'DDoS / DoS', '6': 'Replay Attack'}
  alpha: 0.3


#### SECTION 2: DEFINE LSTM-AUTOENCODER ARCHITECTURE

In [6]:
print("SECTION 2: DEFINE LSTM-AUTOENCODER ARCHITECTURE")
print("="*70)

# Multi-Task LSTM Autoencoder matching the trained model
class MultiTaskLSTMAutoencoder(nn.Module):
    
    def __init__(self, seq_len, num_features, latent_dim, num_classes, hidden_dim=128):
        super(MultiTaskLSTMAutoencoder, self).__init__()
        self.seq_len = seq_len
        self.num_features = num_features
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        
        # Encoder - with more capacity (matching Notebook 2)
        self.encoder_lstm1 = nn.LSTM(num_features, hidden_dim, batch_first=True, bidirectional=True)
        self.encoder_lstm2 = nn.LSTM(hidden_dim * 2, hidden_dim * 2, batch_first=True)
        
        # Project to latent space
        self.latent_proj = nn.Sequential(
            nn.Linear(hidden_dim * 2, latent_dim),
            nn.ReLU()
        )
        
        # Decoder
        self.decoder_lstm1 = nn.LSTM(latent_dim, hidden_dim * 2, batch_first=True)
        self.decoder_lstm2 = nn.LSTM(hidden_dim * 2, hidden_dim, batch_first=True)
        
        # FIX: Add Tanh activation to constrain output to [-1, 1]
        self.decoder_proj = nn.Sequential(
            nn.Linear(hidden_dim, num_features),
            nn.Tanh()
        )
        
        # Classifier - matching Notebook 2 architecture
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Encoder
        encoder_out1, _ = self.encoder_lstm1(x)
        encoder_out2, (hidden, cell) = self.encoder_lstm2(encoder_out1)
        
        # Use last hidden state
        last_hidden = encoder_out2[:, -1, :]
        
        # Project to latent
        latent = self.latent_proj(last_hidden)
        
        # Decoder
        latent_seq = latent.unsqueeze(1).repeat(1, self.seq_len, 1)
        decoder_out1, _ = self.decoder_lstm1(latent_seq)
        decoder_out2, _ = self.decoder_lstm2(decoder_out1)
        recon = self.decoder_proj(decoder_out2)
        
        # Classifier
        class_logits = self.classifier(latent)
        
        return recon, class_logits
    
    def encode(self, x):
        """Extract latent representation"""
        encoder_out1, _ = self.encoder_lstm1(x)
        encoder_out2, (hidden, cell) = self.encoder_lstm2(encoder_out1)
        last_hidden = encoder_out2[:, -1, :]
        return self.latent_proj(last_hidden)


# SECTION 2: DEFINE LSTM-AUTOENCODER ARCHITECTURE
# ====================================================================

print("SECTION 2: DEFINE LSTM-AUTOENCODER ARCHITECTURE")
print("="*70)

# NOTE: These must match the TRAINED model parameters from Notebook 2
# Check your model_config.json for exact values
LSTM_SEQ_LEN = 10
LSTM_FEATURES = 73
LSTM_LATENT_DIM = 32      # MATCH Notebook 2
LSTM_HIDDEN_DIM = 128     # MATCH Notebook 2
LSTM_NUM_CLASSES = 5
LSTM_THRESHOLD = 1.5

# Load the model with the correct architecture
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nLoading LSTM-Autoencoder on device: {device}")

autoencoder = MultiTaskLSTMAutoencoder(
    seq_len=LSTM_SEQ_LEN,
    num_features=LSTM_FEATURES,
    latent_dim=LSTM_LATENT_DIM,
    num_classes=LSTM_NUM_CLASSES,
    hidden_dim=LSTM_HIDDEN_DIM
).to(device)

try:
    # Load the saved state dict
    state_dict = torch.load("../outputs/lstm_autoencoder.pt", map_location=device)
    
    # Check for mismatched keys and handle them
    model_dict = autoencoder.state_dict()
    
    # Filter out mismatched keys (if any)
    filtered_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}
    
    # Load the filtered dict
    autoencoder.load_state_dict(filtered_dict, strict=False)
    autoencoder.eval()
    print("Successfully loaded lstm_autoencoder.pt")
    print(f"Loaded {len(filtered_dict)}/{len(model_dict)} layers")
    
except FileNotFoundError:
    print("ERROR: lstm_autoencoder.pt not found")
    print("Please run Notebook 2 (2-model_trained.ipynb) first.")
    raise

SECTION 2: DEFINE LSTM-AUTOENCODER ARCHITECTURE
SECTION 2: DEFINE LSTM-AUTOENCODER ARCHITECTURE

Loading LSTM-Autoencoder on device: cpu
Successfully loaded lstm_autoencoder.pt
Loaded 47/47 layers


#### SECTION 3: LOAD CROSS-DATASET SAMPLES

In [ ]:
print("\n" + "="*70)
print("SECTION 3: LOADING CROSS-DATASET SAMPLES")
print("="*70)

def load_cross_dataset(npz_filename):
    """
    Load preprocessed cross-dataset samples.
    
    Args:
        npz_filename: Path to NPZ file from Notebook 1 preprocessing
    
    Returns:
        Tuple of (X_ML, y_ML, X_DL, y_DL)
    """
    data = np.load(npz_filename)
    return (
        data["X_test_ML"],
        data["y_test_ML"],
        data["X_test_DL"],
        data["y_test_DL"]
    )

# Dictionary to store all cross-datasets
cross_datasets = {}
failed_datasets = []

# List of datasets to evaluate (modify paths based on your setup)
dataset_configs = [
    {'name': 'Siemens S7 Physical', 'file': 'siemens_s7_processed.npz'},
    {'name': 'OpenPLC Lab (BetterCAP)', 'file': 'openplc_lab_processed.npz'},
    {'name': 'Electra S7Comm', 'file': 'electra_s7comm_processed.npz'},
]

for config in dataset_configs:
    try:
        dataset_name = config['name']
        filename = config['file']
        X_ML, y_ML, X_DL, y_DL = load_cross_dataset(filename)
        cross_datasets[dataset_name] = {
            'X_ML': X_ML,
            'y_ML': y_ML,
            'X_DL': X_DL,
            'y_DL': y_DL,
            'file': filename
        }
        print(f"Loaded: {dataset_name}")
        print(f"  ML shape: {X_ML.shape}, DL shape: {X_DL.shape}")
        print(f"  Class distribution: Normal={np.sum(y_ML==0)}, Attack={np.sum(y_ML==1)}")
    except FileNotFoundError:
        failed_datasets.append(config['name'])
        print(f"Not found: {config['name']} ({config['file']})")

if not cross_datasets:
    print("\n" + "*"*70)
    print("WARNING: No cross-datasets loaded.")
    print("*"*70)
    print("\nTo proceed with cross-dataset evaluation:")
    print("1. Run Notebook 1 on Siemens S7 dataset")
    print("2. Run Notebook 1 on OpenPLC Lab PCAP captures")
    print("3. Run Notebook 1 on Electra S7Comm dataset")
    print("\nThese will generate the required .npz files.")
    print("\nFor now, using simulated cross-domain data for demonstration...\n")
    
    # Generate simulated cross-domain data with intentional domain shift
    np.random.seed(SEED)
    num_samples = 500
    num_features = model_config['num_features']
    seq_len = model_config['seq_len']
    
    # Siemens S7: Different distribution (domain shift)
    X_siemens_ML = np.random.normal(loc=0.3, scale=1.5, size=(num_samples, num_features))
    y_siemens_ML = np.random.choice([0, 1], size=num_samples, p=[0.75, 0.25])
    X_siemens_DL = np.random.normal(loc=0.3, scale=1.5, size=(num_samples, seq_len, num_features))
    y_siemens_DL = y_siemens_ML
    
    # OpenPLC Lab: Different distribution
    X_openplc_ML = np.random.normal(loc=-0.2, scale=1.8, size=(num_samples, num_features))
    y_openplc_ML = np.random.choice([0, 1], size=num_samples, p=[0.8, 0.2])
    X_openplc_DL = np.random.normal(loc=-0.2, scale=1.8, size=(num_samples, seq_len, num_features))
    y_openplc_DL = y_openplc_ML
    
    cross_datasets['Siemens S7 Physical'] = {
        'X_ML': X_siemens_ML,
        'y_ML': y_siemens_ML,
        'X_DL': X_siemens_DL,
        'y_DL': y_siemens_DL,
        'file': 'SIMULATED'
    }
    
    cross_datasets['OpenPLC Lab (BetterCAP)'] = {
        'X_ML': X_openplc_ML,
        'y_ML': y_openplc_ML,
        'X_DL': X_openplc_DL,
        'y_DL': y_openplc_DL,
        'file': 'SIMULATED'
    }
    
    print("Loaded (simulated) cross-datasets for demonstration.")

print(f"\nTotal cross-datasets loaded: {len(cross_datasets)}")

#### SECTION 4: CROSS-DATASET EVALUATION ENGINE

In [ ]:
def evaluate_models_on_dataset(X_ML, y_ML, X_DL, y_DL, dataset_name, threshold):
    """
    Evaluate all three trained models on a cross-dataset sample.
    
    Args:
        X_ML: 2D feature matrix for ML models
        y_ML: Labels for ML models
        X_DL: 3D sequence matrix for DL models
        y_DL: Labels for DL models
        dataset_name: Name of the dataset
        threshold: Anomaly threshold for LSTM-Autoencoder
    
    Returns:
        Dictionary with metrics for all three models
    """
    print(f"\n" + "-"*70)
    print(f"Evaluating on: {dataset_name}")
    print("-"*70)
    
    results = {}
    
    # Isolation Forest predictions
    raw_preds_if = iso_forest.predict(X_ML)
    y_pred_if = np.where(raw_preds_if == -1, 1, 0)
    
    results['Isolation Forest'] = {
        'Accuracy': accuracy_score(y_ML, y_pred_if),
        'Precision': precision_score(y_ML, y_pred_if, zero_division=0),
        'Recall': recall_score(y_ML, y_pred_if, zero_division=0),
        'F1-Score': f1_score(y_ML, y_pred_if, zero_division=0),
        'y_pred': y_pred_if
    }
    
    # XGBoost predictions
    y_pred_xgb = xgb_model.predict(X_ML)
    y_pred_xgb_proba = xgb_model.predict_proba(X_ML)[:, 1]
    
    try:
        roc_auc_xgb = roc_auc_score(y_ML, y_pred_xgb_proba)
    except:
        roc_auc_xgb = 0.0
    
    results['XGBoost'] = {
        'Accuracy': accuracy_score(y_ML, y_pred_xgb),
        'Precision': precision_score(y_ML, y_pred_xgb, zero_division=0),
        'Recall': recall_score(y_ML, y_pred_xgb, zero_division=0),
        'F1-Score': f1_score(y_ML, y_pred_xgb, zero_division=0),
        'ROC-AUC': roc_auc_xgb,
        'y_pred': y_pred_xgb
    }
    
    # LSTM-Autoencoder predictions (handles multi-task output)
    test_tensor = torch.tensor(X_DL, dtype=torch.float32).to(device)
    with torch.no_grad():
        reconstructions, class_logits = autoencoder(test_tensor)           # The autoencoder returns (reconstruction, class_logits) 
        recon_errors = torch.mean((reconstructions - test_tensor) ** 2, dim=(1, 2)).cpu().numpy()
    
    y_pred_ae = np.where(recon_errors > threshold, 1, 0)
    
    results['LSTM-Autoencoder'] = {
        'Accuracy': accuracy_score(y_DL, y_pred_ae),
        'Precision': precision_score(y_DL, y_pred_ae, zero_division=0),
        'Recall': recall_score(y_DL, y_pred_ae, zero_division=0),
        'F1-Score': f1_score(y_DL, y_pred_ae, zero_division=0),
        'y_pred': y_pred_ae,
        'recon_errors': recon_errors
    }
    
    # Print detailed metrics
    for model_name, metrics in results.items():
        print(f"\n{model_name}:")
        for metric, value in metrics.items():
            if metric != 'y_pred' and metric != 'recon_errors':
                print(f"  {metric}: {value:.4f}")
    
    return results

# Perform cross-dataset evaluation
print("\n" + "="*70)
print("SECTION 4: CROSS-DATASET EVALUATION")
print("="*70)

all_results = {}
threshold = model_config['threshold']

for dataset_name, dataset_dict in cross_datasets.items():
    results = evaluate_models_on_dataset(
        dataset_dict['X_ML'],
        dataset_dict['y_ML'],
        dataset_dict['X_DL'],
        dataset_dict['y_DL'],
        dataset_name,
        threshold
    )
    all_results[dataset_name] = results

#### SECTION 5: GENERALIZATION GAP ANALYSIS

In [ ]:
print("\n" + "="*70)
print("SECTION 5: GENERALIZATION GAP ANALYSIS")
print("="*70)

# Build comprehensive results table
results_summary = []

for dataset_name, models in all_results.items():
    for model_name, metrics in models.items():
        row = {
            'Dataset': dataset_name,
            'Model': model_name,
            'Accuracy': metrics.get('Accuracy', 0),
            'Precision': metrics.get('Precision', 0),
            'Recall': metrics.get('Recall', 0),
            'F1-Score': metrics.get('F1-Score', 0)
        }
        results_summary.append(row)

df_summary = pd.DataFrame(results_summary)

print("\nCross-Dataset Performance Summary:")
print(df_summary.to_string(index=False))

# Save summary to CSV
df_summary.to_csv("cross_dataset_evaluation.csv", index=False)
print("\nSaved: cross_dataset_evaluation.csv")

# Pivot table: Models vs Datasets
df_pivot = df_summary.pivot_table(
    index='Model',
    columns='Dataset',
    values='F1-Score',
    aggfunc='mean'
)

print("\n" + "-"*70)
print("F1-Score by Model and Dataset:")
print("-"*70)
print(df_pivot.round(4))

# Calculate generalization gap
if len(cross_datasets) > 0:
    print("\n" + "-"*70)
    print("GENERALIZATION GAP ANALYSIS:")
    print("-"*70)
    
    for model_name in df_pivot.index:
        scores = df_pivot.loc[model_name].values
        gap = np.max(scores) - np.min(scores)
        std = np.std(scores)
        mean = np.mean(scores)
        
        print(f"\n{model_name}:")
        print(f"  Mean F1-Score: {mean:.4f}")
        print(f"  Max F1-Score: {np.max(scores):.4f}")
        print(f"  Min F1-Score: {np.min(scores):.4f}")
        print(f"  Generalization Gap: {gap:.4f}")
        print(f"  Standard Deviation: {std:.4f}")

#### SECTION 6: VISUALIZATION & REPORTING

In [ ]:
print("\n" + "="*70)
print("SECTION 6: VISUALIZATION & REPORTING")
print("="*70)

# Plot 1: F1-Score Comparison Across Datasets
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(df_pivot.columns))
width = 0.25

for idx, model_name in enumerate(df_pivot.index):
    values = df_pivot.loc[model_name].values
    ax.bar(x + idx*width, values, width, label=model_name, edgecolor='black', linewidth=1)

ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Cross-Dataset Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(df_pivot.columns, rotation=15, ha='right')
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig("cross_dataset_f1_comparison.png", dpi=100, bbox_inches='tight')
print("Saved: cross_dataset_f1_comparison.png")
plt.show()

# Plot 2: Heatmap of all metrics
fig, ax = plt.subplots(figsize=(12, 6))

heatmap_data = df_summary.pivot_table(
    index=['Dataset', 'Model'],
    values='F1-Score',
    aggfunc='mean'
)

# Flatten for visualization
pivot_data = df_summary.pivot_table(
    index='Model',
    columns='Dataset',
    values='F1-Score'
)

sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
            cbar_kws={'label': 'F1-Score'}, vmin=0, vmax=1, linewidths=1,
            linecolor='black')
ax.set_title('F1-Score Heatmap: Models vs Datasets', fontsize=14, fontweight='bold')
ax.set_xlabel('Dataset', fontsize=12, fontweight='bold')
ax.set_ylabel('Model', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig("f1_score_heatmap.png", dpi=100, bbox_inches='tight')
print("Saved: f1_score_heatmap.png")
plt.show()

# Plot 3: Generalization Gap Visualization
fig, ax = plt.subplots(figsize=(12, 6))

gap_data = []
for model_name in df_pivot.index:
    scores = df_pivot.loc[model_name].values
    gap = np.max(scores) - np.min(scores)
    gap_data.append({'Model': model_name, 'Gap': gap})

df_gap = pd.DataFrame(gap_data).sort_values('Gap', ascending=True)

colors = ['#2ecc71' if gap < 0.2 else '#f39c12' if gap < 0.4 else '#e74c3c' 
          for gap in df_gap['Gap']]

ax.barh(df_gap['Model'], df_gap['Gap'], color=colors, edgecolor='black', linewidth=1.5)
ax.set_xlabel('Generalization Gap (F1-Score Variation)', fontsize=12, fontweight='bold')
ax.set_title('Model Generalization Gaps Across Datasets\n(Lower is Better)', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.5)

for idx, (model, gap) in enumerate(zip(df_gap['Model'], df_gap['Gap'])):
    ax.text(gap + 0.01, idx, f'{gap:.4f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig("generalization_gap.png", dpi=100, bbox_inches='tight')
print("Saved: generalization_gap.png")
plt.show()